# SmartSpend — Personal Expense Intelligence
## Day 2: Load and understand your real dataset

This notebook uses your uploaded `expense_data_1.csv` file.

**How to use in Google Colab**
1. Upload `expense_data_1.csv` using the folder icon in the left sidebar.
2. Run the cells from top to bottom using **Shift + Enter**.
3. Read the notes before running each cell.

The notebook does not assume your column names; it first inspects the dataset so you can adapt the analysis to its actual structure.


## 1. Import libraries
Pandas helps us work with tables. Matplotlib and Seaborn help us make charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("SmartSpend started!")


## 2. Load the CSV
The filename below matches the uploaded file. If you upload a differently named file to Colab, change `file_path` accordingly.

In [ ]:
file_path = "/content/expense_data_1.csv"
df = pd.read_csv(file_path)

print("Dataset shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())


## 3. Understand the dataset
These checks show column types, missing values, duplicates, and basic statistics.

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumeric summary:")
display(df.describe(include="number").T)


## 4. Inspect text/category columns
This helps identify possible category, payment method, date, and description fields without guessing their names.

In [ ]:
print("Possible categorical/text columns:")
for col in df.select_dtypes(include=["object", "string"]).columns:
    print(f"\\n{col}: {df[col].nunique()} unique values")
    display(df[col].value_counts(dropna=False).head(10).to_frame("count"))


## 5. Identify likely date and amount columns
Column names vary across datasets. This helper suggests likely candidates; inspect the printed lists and choose the correct columns in the next cell.

In [ ]:
date_candidates = [c for c in df.columns if any(word in c.lower() for word in ["date", "time", "day"])]
amount_candidates = [c for c in df.columns if any(word in c.lower() for word in ["amount", "expense", "spent", "price", "cost", "debit", "value"])]

print("Possible date columns:", date_candidates)
print("Possible amount columns:", amount_candidates)


## 6. Configure columns
After running the previous cell, replace the values below with the exact column names from your dataset. Use `None` if a matching column does not exist. This prevents us from silently assuming the wrong fields.

In [ ]:
# Edit these values after inspecting the actual column names.
DATE_COL = date_candidates[0] if date_candidates else None
AMOUNT_COL = amount_candidates[0] if amount_candidates else None

print("Selected date column:", DATE_COL)
print("Selected amount column:", AMOUNT_COL)


## 7. First expense summary
This cell runs only when an amount column was identified. Confirm that the selected column really represents transaction amounts before relying on the results.

In [ ]:
if AMOUNT_COL is not None:
    df[AMOUNT_COL] = pd.to_numeric(df[AMOUNT_COL], errors="coerce")
    print("Rows:", len(df))
    print("Missing amount values:", df[AMOUNT_COL].isna().sum())
    print("Total of amount column:", df[AMOUNT_COL].sum())
    print("Average amount:", round(df[AMOUNT_COL].mean(), 2))
    print("Highest amount:", df[AMOUNT_COL].max())
else:
    print("No likely amount column found. Check df.columns.tolist() and set AMOUNT_COL manually.")


## 8. Optional category-wise bar chart
Set `CATEGORY_COL` to the exact category column name after inspecting the columns. If there is no category field, skip this cell.

In [ ]:
# Example: CATEGORY_COL = "Category"
CATEGORY_COL = None  # Replace None with the exact category column name.

if AMOUNT_COL is not None and CATEGORY_COL is not None and CATEGORY_COL in df.columns:
    category_spending = (
        df.groupby(CATEGORY_COL)[AMOUNT_COL]
          .sum()
          .sort_values(ascending=False)
    )
    category_spending.plot(kind="bar", figsize=(10, 5))
    plt.title("Spending by Category")
    plt.xlabel("Category")
    plt.ylabel("Amount")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Set CATEGORY_COL to a valid category column to make this chart.")


## Day 2 checkpoint

Before cleaning the data, answer these questions from the outputs above:
1. How many rows and columns are in the dataset?
2. Which column appears to contain transaction amounts?
3. Are there missing values or duplicate rows?
4. Which column could represent a category?

**Important:** Do not delete negative values automatically. In real financial data, they may represent refunds or reversals. First inspect what they mean.
